# <center>ECON 525: Data Management & Visualization in Economics <a class="tocSkip"></center>    
# <center>Python - Introduction to SQLite  <a class="tocSkip"></center>

## Learning Objectives
<hr>

- Understand the basics of SQL Lite
- Learn how to connect to a SQL Lite database
- Learn how to create tables in a SQL Lite database
- Learn how to insert data into a SQL Lite database
- Learn how to query data from a SQL Lite database
- Learn how to update data in a SQL Lite database
- Learn how to delete data from a SQL Lite database


## Introduction
<hr>

When you load a CSV file in Python and assign it to a variable, it consumes your computer's memory. Accessing data from a SQL database is more efficient and enables you to import only the necessary portions of the data.

In this lesson, we will explore some methods to achieve this. This is our first introduction to SQL databases. We will use the SQLite database, which is a lightweight database that is easy to set up and use. SQLite is a self-contained, serverless, zero-configuration, transactional SQL database engine. It is the most widely deployed database in the world with applications in many devices and operating systems.

### What is SQL?
SQL stands for Structured Query Language. It is a standard language for storing, manipulating, and retrieving data in databases. SQL is used to communicate with a database. According to ANSI (American National Standards Institute), it is the standard language for relational database management systems. SQL statements are used to perform tasks such as update data on a database, or retrieve data from a database. This is our first introduction to SQL databases. We are going to explore more about this topic throughout the course as we progress.

### The sqlite3 module
The [sqlite3](https://docs.python.org/3/library/sqlite3.html) module provides a straightforward interface for interacting with SQLite databases. A connection object is created using `sqlite3.connect()`; the connection must be closed at the end of the session with the `.close()` command. While the connection is open, any interactions with the database require you to make a cursor object with the `.cursor()` command. The cursor is then ready to perform all kinds of operations with `.execute()`.

In [ ]:
import sqlite3
import pandas as pd

In [ ]:
# create empty database
connection = sqlite3.connect("gta.db")
# communicate with the database
cursor = connection.cursor()

In [ ]:
# data to be inserted into the database
release_list = [
    (1997, "Grand Theft Auto", "state of New Guernsey"),
    (1999, "Grand Theft Auto 2", "Anywhere, USA"),
    (2001, "Grand Theft Auto III", "Liberty City"),
    (2002, "Grand Theft Auto: Vice City", "Vice City"),
    (2004, "Grand Theft Auto: San Andreas", "state of San Andreas"),
    (2008, "Grand Theft Auto IV", "Liberty City"),
    (2013, "Grand Theft Auto V", "Los Santos")
]

In [ ]:
city_list = [
    ("Liberty City", "New York"),
    ("state of New Guernsey", "state of New Jersey"),
    ("Anywhere, USA", "all USA cities"),
    ("Vice City", "Miami"),
    ("state of San Andreas", "state of California"),
    ("Los Santos", "Los Angeles")
]

In [ ]:
# Drop the table (replace 'table_name' with actual table name)
cursor.execute("DROP TABLE IF EXISTS gta")

# create database table and populate it with release_list
cursor.execute("create table gta (release_year integer, release_name text, city text)")
cursor.executemany("insert into gta values (?,?,?)", release_list)
# save changes immediatley
connection.commit()

In [ ]:
# print all the rows from the gta table
for row in cursor.execute("select * from gta"):
    print(row)

In [ ]:
# print specific rows from the gta table
print("******************************")
cursor.execute("select * from gta where city=:c", {"c": "Liberty City"})
gta_search = cursor.fetchall()
print(gta_search)

In [ ]:
type(gta_search)

In [ ]:
# print the columns of the gta table
cursor.execute("PRAGMA table_info(gta)") # special command that retrieves metadata about the columns of a table
columns = cursor.fetchall()
print(columns)

In [ ]:
cursor.execute("DROP TABLE IF EXISTS cities")

# create a new cities table
cursor.execute("create table cities (gta_city text, real_city text)")
cursor.executemany("insert into cities values (?,?)", city_list)
# save changes immediatley
connection.commit()

In [ ]:
# print all the rows from the gta table
for row in cursor.execute("select * from cities"):
    print(row)

In [ ]:
# select only the rows where gta_city=Liberty City
cursor.execute("select * from cities where gta_city=:c", {"c": "Liberty City"})
cities_search = cursor.fetchall()
print(cities_search)

In [ ]:
gta_search

In [ ]:
# combine data from 2 tables and manipulate it
print("***************************************")
for i in gta_search:
    #replace every instance of "Liberty City" with "New York"
    adjusted = [cities_search[0][1] if value==cities_search[0][0] else value for value in i]
    print(adjusted)

In [ ]:
connection.close()

## Relational Databases
<hr>


* A relational database organizes its data into *tables*, sometimes called *relations*.
* Each table is made up of a fixed number of *columns* and zero or more *rows*
* Each column has a name, and generally has a data type (SQLite is a bit lenient on data types, but a good database design keeps a uniform type on every row in a column)
* A table usually has a *primary key*—a column or set of columns that is used to uniquely identify a row. Each row primary key must be unique througout that table.

So let's get started by importing some data. Let's say we have a CSV file that holds students and their grades in certain courses:

```
studentid,course,grade,credits
1,PY205,3.8,3
1,PY411,3.2,4
1,PY412,3.1,4
1,E101,3.8,2
1,CS216,3.5,3
2,PY205,3.1,3
2,PY411,2.8,4
2,PY412,2.4,4
2,E101,2.9,2
2,CS216,3.3,3
```
Both students took the same courses. Student 1 did pretty well, student 2 not as well. Grades are assigned a numerical value with A being a 4, B being a 3, etc.

If this data is in a file called ``grades.csv`` then we can import it with the following commands:

In [ ]:
# import grades.csv  in a sqlite database

# Load the CSV data
csv_file_path = 'grades.csv'
data = pd.read_csv(csv_file_path)

In [ ]:
data.head(10)

In [ ]:
# Connect to SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect('students_grades.db')

# Save the DataFrame to an SQLite table
data.to_sql('grades', conn, if_exists='replace', index=False)

# Commit and close the connection
conn.commit()
conn.close()

print("Data imported into the SQLite database successfully!")

Let's start exploring our new database.     

Check out the [SELECT documentation](https://www.sqlite.org/lang_select.html) and take a look at the syntax flowchart. It's not important to understand it all now, but keep that flowchart handy as we learn more concepts.

The important bits to notice are the different clauses:
* **SELECT** - lists the columns to return in the results, or ``*`` for all columns. These can actually be any expression, which you can use to compute values, as we'll see in a bit.
* **FROM** - defines which tables to use as input to the query. If you have more than one table, then the tables are joined. More on joins later.
* **WHERE** - Defines conditional expressions that filter the rows from the input tables
* **GROUP BY** - Used in aggregate queries. More on this later.
* **HAVING** - Expressions used to filter groups on an aggregate query.
* **ORDER BY** - Expressions used to sort the results before returning them
* **LIMIT** - Places limits or returns a slice of the results.

In [ ]:
# Install this kernel to use the sqlite3 module
# %pip install ipython-sql

Once installed, you can load the SQL extension and connect it to your SQLite database:

In [ ]:
%load_ext sql
%config SqlMagic.style = 'DEFAULT'


In [ ]:
# Connect to the grades database
%sql sqlite:///students_grades.db


Now, you can write SQL queries directly in the notebook using %sql or %%sql magic commands:

In [ ]:
# Do this if you get an error:
# %pip uninstall -y ipython-sql prettytable
# %pip install -U ipython-sql "prettytable<3"

In [ ]:
# Example 1: Display the first few rows from the grades table
%sql SELECT * FROM grades LIMIT 7;

In [ ]:
# Example 2: Find all students who have a grade higher than 3.5
%sql SELECT * FROM grades WHERE grade > 3.5;

•	Use %sql for a single-line SQL query.

•	Use %%sql for multi-line queries. Example:

In [ ]:
%%sql
SELECT course, AVG(grade) as avg_grade
FROM grades
GROUP BY course;

#### Filtering rows with a WHERE clause
You can filter rows using a WHERE clause in your query. You can use simple comparison expressions here. You can have multiple conditions separated by ``AND`` and ``OR`` keywords.

In [ ]:
%sql SELECT * FROM grades WHERE studentid=1;

In [ ]:
%sql SELECT * FROM grades WHERE course='PY205';

In [ ]:
%sql SELECT * FROM grades WHERE studentid=2 AND course='E101';

In [ ]:
%sql SELECT * FROM grades WHERE grade>3.5;

In [ ]:
%sql SELECT * FROM grades WHERE studentid=2 AND grade>3.0;

#### Expressions and the SELECT clause

You can list the specific columns you want in your results in the `SELECT` clause, separated by commas. In fact, you can include any expression you want there, including computations that involve multiple columns.

From the [SQLite documentation](https://www.sqlite.org/lang_select.html#resultset):

> The list of expressions between the `SELECT` and `FROM` keywords is known as the result expression list. If a result expression is the special expression `"*"`, then all columns in the input data are substituted for that one expression.

In [ ]:
%sql SELECT studentid, grade FROM grades;

This example shows computing a column from an expression. The grade points earned for a course is the grade earned times the credits. This quantity is used in computing grade point averages.

In [ ]:
%sql SELECT studentid, course, grade, credits, grade*credits FROM grades;

You can change the result column name by giving an alternative name like so. This is useful for clarity, but also you can refer to these names later in the query in the WHERE clause and a few other clauses.

In [ ]:
%sql SELECT studentid, course, grade*credits AS gradepoints FROM grades;

Finally, take a look at the reference page on [Core Functions](https://www.sqlite.org/lang_corefunc.html) and the reference page on [Expressions](https://www.sqlite.org/lang_expr.html). The core functions can be used in any expression, and the expression syntax has other features. Here are two examples.

Using a CASE expression to transform numerical grades to letters:

In [ ]:
%%sql SELECT studentid, course, grade,
       CASE WHEN grade<0.66 THEN 'F'
            WHEN grade<1.66 THEN 'D'
            WHEN grade<2.66 THEN 'C'
            WHEN grade<3.66 THEN 'B'
            ELSE 'A' END
                AS lettergrade
    FROM grades;

Using a LIKE expression in the WHERE clause to match all Physics classes. LIKE expressions do pattern matching using the percent sign as a wildcard.

In [ ]:
%sql SELECT * FROM grades WHERE course LIKE 'PY%';

Now you know how to do simple selects and queries on a table, filtering results and doing some simple computations. Take some time to play with various expressions and queries by your own!   

#### Grouping and Aggregate Queries
So you can now select rows from the table and compute expressions per-row, but what if you want to compute a student's GPA from all their courses? Because each (student,course) pair is its own row, we can't computer a GPA because that involves data from more than one row.

SQL provides a grouping mechanism that allows you to do aggregate computations on each group. This is done with the GROUP BY clause, which goes after the FROM and WHERE clauses.

The GROUP BY clause takes one or more expressions, separated by commas, that define the groups. Typically, these expressions will be column names but you can compute values with expressions, too.

SQL will then take each row and assign it a group based on the GROUP BY expression(s). **The results of the query will be one row per group.** From the [SQLite docs](https://www.sqlite.org/lang_select.html#resultset):
> [...] each of the expressions specified as part of the GROUP BY clause is evaluated for each row of the dataset. Each row is then assigned to a "group" based on the results; rows for which the results of evaluating the GROUP BY expressions are the same get assigned to the same group.

Using a GROUP BY clause makes the query into an *aggregate query*. In your SELECT clause, you can then use one or more [aggregate functions](https://www.sqlite.org/lang_aggfunc.html). The aggregate functions operate over each row in the group. So, for example, you can compute the average grade per student like this:

In [ ]:
%sql SELECT studentid, AVG(grade) AS avg_grade FROM grades GROUP BY studentid;

Now we know enough to use grouping to compute each student's GPA:

In [ ]:
%sql SELECT studentid, SUM(grade*credits)/SUM(credits) AS GPA FROM grades GROUP BY studentid;

This takes the total gradepoints of each group and divides them by the total credits in that group. Since the grouping is by studentid, each student is a group, and each group contains the rows for that student. So any aggregate functions operate on the rows for each student independently.

If you combine the WHERE clause with an aggregate query, the WHERE clause filters the input rows before grouping. So here is the GPA of all students just for their Physics classes:

In [ ]:
%sql SELECT studentid,  SUM(grade*credits)/SUM(credits) AS GPA FROM grades WHERE course LIKE 'PY%' GROUP BY studentid;

If you add a HAVING clause, this filters the groups. It's essentially like the WHERE clause except it applies *after* grouping instead of before. You can use aggregate expressions in the HAVING clause.

This selects students that have a GPA greater than 3. Notice how we don't actually select the GPA into the results, we just get the studentids, and calculate the GPA in the HAVING clause.

In [ ]:
%sql SELECT studentid FROM grades GROUP BY studentid HAVING SUM(grade*credits)/SUM(credits) >= 3;

In [ ]:
# Check the difference between using WHERE and HAVING clauses in SQL. The WHERE clause filters rows before aggregation, while the HAVING clause filters groups after aggregation.
%sql SELECT studentid, SUM(grade*credits)/SUM(credits) AS GPA FROM grades GROUP BY studentid  HAVING course LIKE 'PY%' ;

You can also refer to a column name in other clauses. So if you *do* want the GPA in the results, to avoid having to duplicate the expression in both clauses, give it a name and refer to it by name:

In [ ]:
%%sql SELECT studentid, SUM(grade*credits)/SUM(credits) AS GPA 
    FROM grades
    GROUP BY studentid
    HAVING GPA >= 3;

#### Two more notes on Aggregate Queries

**1.** if you use an *aggregate function* but don't use a GROUP BY clause, the query is still an aggregate query, but with one large group containing every row.

This is the total GPA of all students, which may or may not be meaningful.

In [ ]:
%sql SELECT SUM(grade*credits)/SUM(credits) AS GPA FROM grades;

**2.** If you select on a column that is neither an aggregate function nor a column named in the GROUP BY, you're essentially asking about a column for which there may be multiple results. This is allowed, but since the results only give one row per group, SQLite will pick an arbitrary row within the group to return, which is usually not meaningful.

From the [SQLite docs](https://www.sqlite.org/lang_select.html#resultset):
> Each expression in the result-set is then evaluated once for each group of rows. If the expression is an aggregate expression, it is evaluated across all rows in the group. Otherwise, it is evaluated against a single arbitrarily chosen row from within the group. If there is more than one non-aggregate expression in the result-set, then all such expressions are evaluated for the same row.


For example, this query groups by students and also selects the course column. But since each student takes more than one course, this will return an arbitrary course for each student.

In [ ]:
%sql SELECT studentid, course FROM grades

In [ ]:
%sql SELECT studentid, course FROM grades GROUP BY studentid;

### Joins

**Today’s question:** We have course grades in one table and student information in another. How can we combine them without losing people?

Imagine two lists for an event: a registration list and a list of completed check-ins. Matching them answers “Who checked in?” Keeping everyone registered also answers “Who has not checked in yet?” The choice of join changes the question and the people represented in your analysis.

By the end, you should be able to explain a join key, translate `g.studentid` into words, choose between an inner and a left join, and recognize why missing grades are different from a GPA of zero.

Run the notebook from the top. We will use the `grades` table created earlier and add student information from `demographics.csv`. Activity cells are intentionally blank; later examples do not depend on your answers.


#### 1. Think in tables before writing SQL

![Database relationship: one demographics record links by studentid to many course records.](teaching-assets/sqlite-database-map.png)

In `demographics`, one row describes **one student**. In `grades`, one row describes **one student in one course**. Student 1 appears in several grade rows; that is expected, not a duplicate error. The pair `(studentid, course)` identifies a grade record in this example.

The **join key** is the column used to match records: here, `studentid`. Matching names would be less reliable because people can share names or spell them differently. A relationship is a connection between values; a join builds a result table using that relationship. It does not permanently merge or change the input tables.

**Quick prediction, before running:** The demographic table has a student with ID 3. Will that student necessarily appear when we match only students who have recorded grades?

We now add a second table. `join_conn` opens our existing database; `to_sql` writes the new table. `if_exists="replace"` lets us rerun this cell without appending the same students again.


In [ ]:
# Add student information to the grades database we created earlier.
demographics = pd.read_csv("demographics.csv")
join_conn = sqlite3.connect("students_grades.db")
demographics.to_sql("demographics", join_conn, if_exists="replace", index=False)

demographics


#### From cursors and SQL magic to pandas

Earlier, we used `cursor.execute(...)` and `fetchall()`, then `%sql` / `%%sql`. These are different ways to send SQL to the database. For the joins below, we will use **`pd.read_sql_query(query, join_conn)`**: it runs a `SELECT` and returns a **DataFrame**, making the result easy to inspect and analyze.

**Remember: connect → execute → commit changes → close.**

1. **Connect:** `sqlite3.connect(...)` opens the connection, such as `join_conn` above.
2. **Execute:** send the SQL; here, pandas runs the query and collects its rows.
3. **Commit changes:** use `connection.commit()` after manual `INSERT`, `UPDATE`, or `DELETE` operations. A `SELECT` only reads, so it needs no commit. Our earlier `to_sql(...)` call saves its SQLite inserts for us.
4. **Close:** call `connection.close()` when finished; we close our connections at the end.

Pandas manages the cursor internally, so we do not create one for these queries. Use an explicit cursor when you want to execute SQL and fetch rows yourself, or send writes with `execute` / `executemany`, as in the HR example later. SQLite also offers `connection.execute(...)` as a shortcut that creates a cursor for you.


#### 2. INNER JOIN: keep the matching pairs

“Add age to every recorded course grade, if we know the student.” Start with a grade row, find the demographic row with the same `studentid`, and place their information side by side. If there is no match, an **inner join leaves that row out**.

Read the SQL below in this order: **FROM** chooses the starting table → **INNER JOIN … ON** finds matching rows → **SELECT** chooses the result columns → **ORDER BY** makes the result easy to inspect. SQL is written with `SELECT` first, even though the matching step helps us reason about it.

| Code | Say it in plain English |
|---|---|
| `FROM grades AS g` | Use `grades`, and call it `g` inside this query. This temporary nickname is a **table alias**. |
| `demographics AS d` | Call the other table `d`. `AS` does not rename the stored table. |
| `g.studentid` | The `studentid` column **from `grades`**, whose nickname is `g`. The dot identifies where the column comes from. |
| `d.studentid` | The same-named column from `demographics`. |
| `ON g.studentid = d.studentid` | A grade row and a student row match when their IDs are equal. |
| `g.*` | Every column from `grades` only. It does not mean every row. |
| `g.*, d.*` | All columns from both tables; this includes two `studentid` columns. Naming the columns explicitly makes the result clearer. |

These aliases exist only inside the SQL string. They are not Python variables. `g` and `d` are convenient choices, not special SQL keywords.


In [ ]:
inner_query = """
SELECT g.studentid, g.course, g.grade, d.age  -- choose four result columns
FROM grades AS g                           -- nickname grades as g
INNER JOIN demographics AS d               -- nickname demographics as d
    ON g.studentid = d.studentid            -- the rule for matching rows
ORDER BY g.studentid, g.course;             -- sort the displayed result
"""

# Python sends the SQL string to SQLite and receives a pandas table.
inner_result = pd.read_sql_query(inner_query, join_conn)
print(inner_result.to_string(index=False))


The `--` text is a **SQL comment**, so it belongs inside the SQL string. Outside the string, `#` starts a **Python comment**. Triple quotes let Python store a SQL statement across several lines; they do not execute it. Pandas stores the returned DataFrame in `inner_result`; the final line prints it.

Student 3 is absent: there is no course record for ID 3. We have one result per matched student-course, not one result per student. A join can therefore return more rows than the student table contains.


#### Your turn — SQL-A1 · alone · 5 minutes

An adviser wants the **age and grade of students who have a recorded grade in `PY205`**.

1. Before coding, write your predicted number of rows and say whether student 3 should appear.
2. Write an inner join that returns `studentid`, `age`, and `grade`, keeping only `PY205`. Use a `WHERE` clause after the `ON` condition and sort by student ID.
3. Explain `g.studentid` to your neighbor in one sentence when time is called.

**Prediction and reason:** …  
**My explanation of `g.studentid`:** …

Start with the question and the columns you need. If stuck, identify the two table aliases and the matching key before writing the query.


In [ ]:
# Your turn: write your code here.
# You can use pd.read_sql_query(your_query, join_conn).


#### 3. LEFT JOIN: keep everyone on the chosen starting list

Now ask: **“Which students do we know about, including students without grades?”** Put `demographics` on the **left**, after `FROM`. A `LEFT JOIN` keeps each left-side row. When a right-side match is missing, SQLite fills those right-side fields with `NULL`.

`LEFT JOIN` and `LEFT OUTER JOIN` mean the same thing. “Outer” describes preserving unmatched rows. It does not mean “only unmatched rows.” A left join includes matches too.

Let’s use a tiny, separate example to see the row matching clearly. Ana has two results, Cam has none, and a result with student ID 4 has no roster record. These are teaching examples, separate from our original CSVs.


In [ ]:
visual_conn = sqlite3.connect(":memory:") # create an in-memory -temporary- database, not saved to disk
visual_conn.execute("CREATE TABLE roster (studentid INTEGER PRIMARY KEY, name TEXT)")
visual_conn.execute("CREATE TABLE results (studentid INTEGER, course TEXT, score INTEGER)")
visual_conn.executemany("INSERT INTO roster VALUES (?, ?)", [(1, "Ana"), (2, "Bo"), (3, "Cam")])
visual_conn.executemany("INSERT INTO results VALUES (?, ?, ?)",
                        [(1, "SQL", 92), (1, "Python", 88), (2, "SQL", 85), (4, "SQL", 76)])
visual_conn.commit()

left_query = """
SELECT r.studentid AS roster_studentid, r.name,
       s.studentid AS results_studentid, s.course, s.score
FROM roster AS r
LEFT JOIN results AS s ON r.studentid = s.studentid
ORDER BY r.studentid, s.course;
"""
print(pd.read_sql_query(left_query, visual_conn).to_string(index=False))


![Inputs and actual result rows for inner, left, right and full outer joins. Ana produces two matches; Cam and ID 4 illustrate the unmatched sides.](teaching-assets/sqlite-join-row-map.png)

For the diagram, row order is illustrative; use `ORDER BY` whenever order matters.

| Question | Join to choose | What is preserved? |
|---|---|---|
| Which records match? | `INNER JOIN` | Matching row pairs |
| Keep our whole roster, even without results | `roster LEFT JOIN results` | All roster rows |
| Keep all results, even without roster information | `roster RIGHT JOIN results` | All results rows; equivalently reverse the inputs and use `LEFT JOIN` |
| Audit missing information on either side | `FULL OUTER JOIN` | All rows from both inputs |

**Pause and point:** Find Cam in each result. Find ID 4. Where did Ana’s second row come from? Count matched **pairs**, not just distinct IDs.

`NULL` means unknown or missing. It is not zero and is not the string `"NULL"`. In pandas, numeric missing values may display as `NaN` and text missing values as `None`; those displays represent the SQL missing values returned by this query. ID 4 is absent from a roster-left join because preserving the left does not automatically preserve the right.

We will practice **INNER** and **LEFT** first. Native `RIGHT` and `FULL OUTER JOIN` require SQLite **3.39.0 or later**; the optional extension below uses that syntax. On an older version, reverse table order and use a left join for the right-join question. See [SQLite’s join rules](https://www.sqlite.org/lang_select.html#joinclause) and [the SQLite 3.39.0 release note](https://www.sqlite.org/releaselog/3_39_0.html).


#### Your turn — SQL-A2 · pairs · 5 minutes

Return to `join_conn`, our original student database. An adviser wants **every student in `demographics`**, together with each course and grade when available. Students without a course must still appear.

- Partner A states which table goes after `FROM` and why. Partner B writes the query; switch roles halfway through.
- Return `studentid`, `age`, `course`, and `grade`. Use aliases and sort by the demographic student ID, then course.
- Check the number of rows and inspect the row for student 3. Is “no grade” the same as failing?

**Our table order and reason:** …  
**Our row-count check:** …  
**What the missing grade means:** …


In [ ]:
# Your turn: write your code here.
# You can use pd.read_sql_query(your_query, join_conn).


#### 4. Joins and grouping: one row per student

Our joined table has one row per student-course. An adviser may instead want one row per student. **Join first, group second.**

The credit-weighted GPA is `SUM(grade * credits) / SUM(credits)`. A four-credit class contributes more than a two-credit class. Because grades contain decimal values here, this division produces a decimal answer. `ROUND(..., 2)` rounds it for display.

The worked example uses an **inner join**, so it summarizes only students with a matching course record. `GROUP BY` lists the student attributes we keep alongside the aggregate. Do not include an arbitrary ungrouped course name in a student summary: a student can have several courses.


In [ ]:
gpa_query = """
SELECT d.studentid, d.gender, d.age,
       ROUND(SUM(g.grade * g.credits) / SUM(g.credits), 2) AS GPA
FROM demographics AS d
INNER JOIN grades AS g ON d.studentid = g.studentid
GROUP BY d.studentid, d.gender, d.age
ORDER BY d.studentid;
"""
print(pd.read_sql_query(gpa_query, join_conn).to_string(index=False))


With a left join, a student without courses gets one preserved row whose grade fields are `NULL`. `SUM` ignores missing values; if it has no nonmissing values to add, its result is `NULL`. Their GPA should remain missing, not become zero.

For a count of recorded grades, use **`COUNT(g.grade)`**, which counts only nonmissing grades. **`COUNT(*)` counts result rows**, including the preserved row with no grade. Group by the key of the table you preserve: here, **`d.studentid`**. Grouping only by `g.studentid` would combine all unmatched students into a single missing-key group.


#### Your turn — SQL-A3 · groups of 3 · 5 minutes

Produce one row for **every student in `demographics`**, showing `studentid`, `age`, the number of recorded grades as `n_grades`, and credit-weighted `GPA` rounded to 2 decimals.

Assign roles: **question translator** chooses the join, **coder** writes the query, and **skeptic** checks student 3 and the grouping key. Everyone must be able to explain the result.

1. Adapt the GPA example so it retains students without grades.
2. Add a count that reports **zero recorded grades** for student 3.
3. Explain why replacing their missing GPA with zero would mislead an adviser.

**Join and grouping key:** …  
**Why our count works:** …  
**One sentence for the adviser about student 3:** …


In [ ]:
# Your turn: write your code here.
# You can use pd.read_sql_query(your_query, join_conn).

#### Check your understanding

Complete these sentences without looking back:

- “I use an inner join when …”
- “I start with the roster and use a left join when …”
- “`g.*` means …; `g.studentid` means …”
- “After a left join, missing GPA tells me …”

**Bridge to web scraping:** SQL helps organize and connect structured records. Web scraping helps us turn information on a web page into structured records that we could later put in a table. In both cases, ask what one row represents and which records are missing.

The optional extensions below offer further practice with these ideas.


### Optional extension A — filters can change who remains

Suppose we want every student, but attach only their **Python** course result. There are two different questions:

- Put `s.course = 'Python'` **inside `ON`**: decide which results qualify as matches, then keep the entire roster.
- Put it **in `WHERE`** after the left join: keep only result rows satisfying that condition. Rows without a Python result have `NULL` there and are filtered out.

Predict whether Cam and Bo remain in each result before running. `NULL = 'Python'` does not evaluate to true, so a `WHERE` condition like this removes an unmatched row.


In [ ]:
python_in_on = """
SELECT r.studentid, r.name, s.course, s.score
FROM roster AS r
LEFT JOIN results AS s
    ON r.studentid = s.studentid AND s.course = 'Python'
ORDER BY r.studentid;
"""
python_in_where = """
SELECT r.studentid, r.name, s.course, s.score
FROM roster AS r
LEFT JOIN results AS s ON r.studentid = s.studentid
WHERE s.course = 'Python'
ORDER BY r.studentid;
"""
print("Filter in ON — keep the roster:")
print(pd.read_sql_query(python_in_on, visual_conn).to_string(index=False))
print("\nFilter in WHERE — keep qualifying result rows:")
print(pd.read_sql_query(python_in_where, visual_conn).to_string(index=False))


### Optional extension B — RIGHT, FULL OUTER and CROSS JOIN

**Right join:** keep all rows from the table written on the right. We can ask the same question on older SQLite versions by reversing table order and using a left join. This is about position, not a permanent “left table” or “right table” label.

**Full outer join:** preserve both sides. In our tiny example, Cam and the result for ID 4 both remain. We select both source IDs so a missing roster ID does not hide the ID that exists in `results`.

Check `sqlite3.sqlite_version` if you are unsure which SQLite version Python uses. The queries below require SQLite 3.39.0 or later. The join diagram explains the same ideas if your version is older.


In [ ]:
right_query = """
SELECT r.studentid AS roster_studentid, r.name,
       s.studentid AS results_studentid, s.course, s.score
FROM roster AS r
RIGHT JOIN results AS s ON r.studentid = s.studentid
ORDER BY s.studentid, s.course;
"""
print(pd.read_sql_query(right_query, visual_conn).to_string(index=False))

full_query = """
SELECT r.studentid AS roster_studentid, r.name,
       s.studentid AS results_studentid, s.course, s.score
FROM roster AS r
FULL OUTER JOIN results AS s ON r.studentid = s.studentid;
"""
print(pd.read_sql_query(full_query, visual_conn).to_string(index=False))


**Cross join:** return every possible pair, with no matching condition. Three roster rows times four result rows give twelve pairs. This can be useful for constructing a grid of combinations, such as every region × every year. It is usually the wrong query when you meant to match people by ID.

An inner join can be understood conceptually as possible pairs filtered by an `ON` condition. That is a reasoning model; it does not mean the database must physically build the whole cross product before executing your query.


In [ ]:
cross_query = """
SELECT r.studentid AS roster_studentid, r.name,
       s.studentid AS results_studentid, s.course, s.score
FROM roster AS r
CROSS JOIN results AS s
ORDER BY r.studentid, s.studentid, s.course;
"""
cross_result = pd.read_sql_query(cross_query, visual_conn)
print(f"CROSS JOIN: {len(cross_result)} possible pairs = 3 roster rows × 4 result rows")
print(cross_result.to_string(index=False))


#### A data-quality question before any join

What should one row represent in each input and in the result? If two tables each have several rows for the same key, every matching combination appears. Two records for one key on the left and three on the right create **six** matched pairs. That may be intended, but it can also inflate totals.

In our course data, several grades per student are expected; several demographic records for the same student would require investigation. Check that demographic IDs identify one row per student before joining. Database constraints such as a declared primary key can enforce uniqueness; importing a CSV alone does not establish that constraint.


### Optional extension C — when tables are in separate database files

The original example stored `grades` and `demographics` in two `.db` files. Joins combine tables, and those tables can be in one database or in an attached database. Keeping both tables in one practice database earlier helped us focus on the join itself.

`ATTACH DATABASE ... AS demo_db` gives another database the nickname `demo_db` for this connection. In **`demo_db.demographics AS d`**, `demo_db` is a **database alias**, `demographics` is the table name, and `d` is a **table alias** used within the query. In **`d.studentid`**, the dot instead points from the table alias to its column.

This demonstration attaches another temporary in-memory database, so it does not create or change the supplied files. For a real file, the attachment statement could be `ATTACH DATABASE 'students_demographics.db' AS demo_db`; that text is an example of a file path, not a required new file for today.


In [ ]:
# Attach a separate, temporary database to the same connection.
join_conn.execute("ATTACH DATABASE ':memory:' AS demo_db")
join_conn.execute("CREATE TABLE demo_db.demographics (studentid INTEGER, gender TEXT, age INTEGER)")
with join_conn:
    join_conn.executemany("INSERT INTO demo_db.demographics VALUES (?, ?, ?)",
                          list(demographics.itertuples(index=False, name=None)))

attached_query = """
SELECT g.studentid, g.course, d.age
FROM grades AS g
INNER JOIN demo_db.demographics AS d ON g.studentid = d.studentid
ORDER BY g.studentid, g.course;
"""
print(pd.read_sql_query(attached_query, join_conn).to_string(index=False))
# Detaching removes this connection's access to the attached database.
join_conn.execute("DETACH DATABASE demo_db")


## Advanced Example: HR Database · optional guided extension

We keep the original idea: a small HR program that creates employee records, searches for them, updates pay, and removes a record. This example connects the Python classes/functions you have seen with SQL. You do not need to memorize class syntax to explain the joins above.

**Two kinds of storage:** an `Employee` object lives in Python memory. A database row lives in SQLite. Creating an object does **not** automatically insert a row. We write functions to move values between those worlds.

**Why a class?** Think of `Employee` as a reusable template for packaging one person’s attributes and related behavior. `Employee("Jane", "Doe", 90000)` creates one object; `Employee("John", "Doe", 80000)` creates another. Each object keeps its own `first`, `last`, and `pay`. A class is not a database table, although the attributes can correspond to table columns.

| Python syntax | What it does here |
|---|---|
| `__init__(self, first, last, pay)` | Runs when we create an object and stores the supplied values. |
| `self` | The particular employee object being constructed or used. |
| `self.pay` | That object’s pay value; different objects can store different values. |
| `@property` | Lets us access a computed value like `emp.email`, without parentheses. |
| `__repr__` | Supplies a useful representation when we print or inspect the object. |

The email addresses below are illustrative values computed from fictional names. They are not real instructor or student contact information.


In [ ]:
class Employee:
    """Package a fictional employee's Python attributes in one object."""

    def __init__(self, first, last, pay):
        # self is the specific object; first/last/pay are the incoming values.
        self.first = first
        self.last = last
        self.pay = pay

    @property
    def email(self):
        # Computed when requested; not an extra column in our database.
        return f"{self.first}.{self.last}@example.com"

    @property
    def fullname(self):
        return f"{self.first} {self.last}"

    def __repr__(self):
        # !r shows quotes around text values, which makes inspection clearer.
        return f"Employee({self.first!r}, {self.last!r}, {self.pay!r})"


emp_1 = Employee("John", "Doe", 80000)
emp_2 = Employee("Jane", "Doe", 90000)
print(emp_2)
print(emp_2.fullname, emp_2.email)


#### A separate, resettable HR database

`sqlite3.connect(":memory:")` opens a temporary database. `hr_conn` is the connection, our communication link; `hr_cursor` runs commands and retrieves result rows. `CREATE TABLE` defines the SQL columns; it does not create Python employee objects.

For this tiny example, we require a unique `(first, last)` pair so an update identifies one row. A real HR system should identify employees using a stable employee ID because people can share names. The example’s rule is a teaching simplification.


In [ ]:
# Rerun this setup and the helper definitions together to reset the HR example.
hr_conn = sqlite3.connect(":memory:")
hr_cursor = hr_conn.cursor()
hr_cursor.execute("""
CREATE TABLE employees (
    first TEXT NOT NULL,
    last TEXT NOT NULL,
    pay INTEGER NOT NULL,
    UNIQUE (first, last)
)
""")


#### Small functions give each database action a name

**DRY** means “Don’t Repeat Yourself.” Instead of repeating an entire SQL statement each time, we put it in a function with a descriptive name.

- `insert_emp(emp)` reads attributes from one Python object and inserts a row.
- `get_emps_by_name(lastname)` selects rows; `fetchall()` returns a list of tuples.
- `update_pay(emp, pay)` updates the row and then updates the Python object’s pay, so the two agree.
- `remove_emp(emp)` deletes the database row; the Python object can still exist afterward.

The `:first`, `:last`, and `:pay` placeholders receive **values** from a dictionary. SQLite handles quoting safely; do not build a query by pasting user-supplied text into SQL. The placeholders are not Python attribute access and do not supply table or column names.

`with hr_conn:` groups a database change into a transaction: commit if it succeeds, roll back if it fails. It **does not close the connection**. We explicitly close it at the end. See [Python’s parameter binding and connection context manager documentation](https://docs.python.org/3/library/sqlite3.html).


In [ ]:
def insert_emp(emp):
    """Insert values from one Employee object into one database row."""
    with hr_conn:  # commit the insert on success; roll back on error
        hr_cursor.execute(
            "INSERT INTO employees (first, last, pay) VALUES (:first, :last, :pay)",
            {"first": emp.first, "last": emp.last, "pay": emp.pay},
        )


def get_emps_by_name(lastname):
    """Return database rows whose last name matches the supplied value."""
    hr_cursor.execute(
        "SELECT first, last, pay FROM employees WHERE last = :last ORDER BY first",
        {"last": lastname},
    )
    return hr_cursor.fetchall()  # a list; each row inside it is a tuple


def update_pay(emp, pay):
    """Change the matching database row, then this Python object's pay."""
    with hr_conn:
        hr_cursor.execute(
            "UPDATE employees SET pay = :pay WHERE first = :first AND last = :last",
            {"first": emp.first, "last": emp.last, "pay": pay},
        )
    emp.pay = pay  # object attributes do not update automatically with SQL


def remove_emp(emp):
    """Remove the matching row; this does not delete the Python object."""
    with hr_conn:
        hr_cursor.execute(
            "DELETE FROM employees WHERE first = :first AND last = :last",
            {"first": emp.first, "last": emp.last},
        )


#### Trace one object into the table

Before running: how many database rows exist after creating two objects? How many after calling the two insert functions? After the insert, search by last name. Reading rows returns database values, not new `Employee` objects.

Run the HR setup again before repeating the insertion demonstration: inserting the same names twice would violate our example’s uniqueness rule.


In [ ]:
insert_emp(emp_1)  # object attributes become values in a new SQL row
insert_emp(emp_2)
print(get_emps_by_name("Doe"))


Now predict the result: Jane’s pay becomes 95000, then John’s row is removed. The `WHERE` clauses limit each change to the intended employee; omitting `WHERE` from an update or delete would affect every row.


In [ ]:
update_pay(emp_2, 95000)
print("After updating Jane:", get_emps_by_name("Doe"))
print("Jane's Python object also has pay:", emp_2.pay)

remove_emp(emp_1)
print("After removing John's database row:", get_emps_by_name("Doe"))
print("John's Python object still exists:", emp_1)


#### Optional practice — SQL-A4 · pairs · 5 minutes

Create a fictional employee named **Alex Kim** with pay **72000**. Insert the object using the helper, retrieve the database rows with last name `Kim`, and print the object’s computed email. Explain which action creates a Python object and which creates a SQL row.

**Object action:** …  
**Database-row action:** …  
**Why `alex.email` has no parentheses:** …

This optional practice connects Python objects to database rows. Rerun HR setup and the helper definitions if you want a fresh table before repeating it.


In [ ]:
# Your turn: write your code here.
# Create an Employee object, use the helpers, and inspect its email.


#### Close the practice connections

Close a connection when finished. Our grades and demographics tables remain in `students_grades.db`; the small visual and HR examples use temporary in-memory databases. The conditional check skips HR cleanup if you did not run that optional example.


In [ ]:
join_conn.close()
visual_conn.close()
if "hr_conn" in globals():
    hr_conn.close()
print("Practice connections closed.")


### Takeaway

Before writing a join, say **what one row means, which key matches records, and whose rows you need to keep**. Then choose columns, check row counts, inspect missing matches, and only afterward summarize.

References: [SQLite SELECT and joins](https://www.sqlite.org/lang_select.html), [SQLite aggregate functions](https://www.sqlite.org/lang_aggfunc.html), [pandas read_sql_query](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html), and [Python sqlite3](https://docs.python.org/3/library/sqlite3.html).
